In [ ]:
#0. 작업 준비
import numpy as np
import torch
import matplotlib.pyplot as plt

from torch.utils import data
from torchvision import datasets,transforms, utils
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
USE_CUDA=torch.cuda.is_available()
DEVICE=torch.device('cuda' if USE_CUDA else 'cpu')

In [ ]:
tr_ds=datasets.FashionMNIST(root='./data/',
                     train=True,
                     download=False,
                     transform=transforms.Compose([transforms.ToTensor()]))

In [ ]:
BATCH_SIZE=60000
tr_ds_loader=torch.utils.data.DataLoader(
    dataset=tr_ds,
    batch_size=BATCH_SIZE,
    shuffle=False
)
img,_=next(iter(tr_ds_loader))
img.shape
img.mean(),img.std()

(tensor(0.2860), tensor(0.3530))

In [ ]:
BATCH_SIZE=64
EPOCHS=10

In [ ]:
# 데이터 수정 (노이즈 삽입)
# 1. 데이터 준비
transform=transforms.Compose([
    #transforms.RandomHorizontalFlip(),#데이터 증강(노이즈삽입)
    transforms.ToTensor(),#입력 데이터 정리
    transforms.Normalize((0.2860,),(0.3530,))
])#데이터 사용 방식 내용 결정
tr_ds_loader=torch.utils.data.DataLoader(
    datasets.FashionMNIST('./data/',
        train=True,
        download=False,
        transform=transform
    ),
    batch_size=BATCH_SIZE,
    shuffle=True
)
tt_ds_loader=torch.utils.data.DataLoader(
    datasets.FashionMNIST('./data/',
        train=False,
        transform=transform
    ),
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [ ]:
#28*28*1
#->24*24*10
#->12*12*10
#    ->8*8*20
#    ->4*4*20=320
#    ->12*12*20
#    ->6*6*20=720
#->28*28*10
#->14*14*10
#    ->10*10*20
#    ->5*5*20=500
#    ->14*14*20
#    ->7*7*20=980

In [ ]:
class 모델(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(1,10,kernel_size=5)
        self.conv2=nn.Conv2d(10,20,kernel_size=5)
        self.conv2_drop=nn.Dropout2d()
        self.fc1=nn.Linear(320,50)
        self.fc2=nn.Linear(50,10)
    def forward(self,x):
        x=F.max_pool2d(self.conv1(x),2)
        x=F.max_pool2d(self.conv2_drop(self.conv2(x)),2)
        x=x.view(-1,320)
        x=F.relu(self.fc1(x))
        x=F.dropout(x,training=self.training)
        x=self.fc2(x)
        return x

In [ ]:
m=모델().to(DEVICE)
opt=optim.SGD(m.parameters(),lr=0.01,momentum=0.5)
scheduler=optim.lr_scheduler.StepLR(opt,step_size=3,gamma=0.1)

In [ ]:
def train(m,tr_ds_loader,opt,epoch):
    m.train()
    for i,(x,y) in enumerate(tr_ds_loader):
        data,target=x.to(DEVICE),y.to(DEVICE)
        opt.zero_grad()
        py=m(data)
        loss=F.cross_entropy(py,target)
        loss.backward()
        opt.step()

        if i%100==0:
            print(f"train epoch{epoch} {i*len(data)}/{len(tr_ds_loader.dataset)}({i*100/len(tr_ds_loader)}%) loss:{loss.item()}")
        

In [ ]:
@torch.no_grad()#서식
def evaluate(m,tt_ds_loader):
    m.eval()
    test_loss=0
    correct=0
    for data,target in tt_ds_loader:
        data,target=data.to(DEVICE),target.to(DEVICE)
        py=m(data)
        test_loss+=F.cross_entropy(py,target,reduction='sum').item()
        pred=py.max(1,keepdim=True)[1]
        correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss/=len(tt_ds_loader.dataset)
    test_accuracy= correct/len(tt_ds_loader.dataset)*100.
    return test_loss,test_accuracy   

In [ ]:
for i in range(1,EPOCHS+1):
    train(m,tr_ds_loader,opt,i)
    scheduler.step()
    test_loss,test_acc=evaluate(m,tt_ds_loader)

    print(f'{i}epoch test_loss:{test_loss},test_accuracy:{test_acc},lr_scheduler{scheduler.get_last_lr()} ')

train epoch1 0/60000(0.0%) loss:2.2395551204681396
train epoch1 6400/60000(10.660980810234541%) loss:1.2854253053665161
train epoch1 12800/60000(21.321961620469082%) loss:1.2398531436920166
train epoch1 19200/60000(31.982942430703623%) loss:0.9714453816413879
train epoch1 25600/60000(42.643923240938165%) loss:0.8687777519226074
train epoch1 32000/60000(53.304904051172706%) loss:1.0619844198226929
train epoch1 38400/60000(63.96588486140725%) loss:1.0105922222137451
train epoch1 44800/60000(74.6268656716418%) loss:0.7676950693130493
train epoch1 51200/60000(85.28784648187633%) loss:0.7859184741973877
train epoch1 57600/60000(95.94882729211088%) loss:0.9343363642692566
1epoch test_loss:0.6377642233848572,test_accuracy:75.26
train epoch2 0/60000(0.0%) loss:1.1560724973678589
train epoch2 6400/60000(10.660980810234541%) loss:0.7525548338890076
train epoch2 12800/60000(21.321961620469082%) loss:0.9427090883255005
train epoch2 19200/60000(31.982942430703623%) loss:0.798268735408783
train epoc

In [3]:
ck_tr=transforms.Compose([transforms.ToTensor()])
tr_ds=datasets.CIFAR10(root='./data/',train=True,download=True,transform=ck_tr)
tr_ds_loader=torch.utils.data.DataLoader(tr_ds,batch_size=50000,shuffle=False)

In [4]:
ck_data=iter(tr_ds_loader)
data,_=next(ck_data)
data

tensor([[[[0.2314, 0.1686, 0.1961,  ..., 0.6196, 0.5961, 0.5804],
          [0.0627, 0.0000, 0.0706,  ..., 0.4824, 0.4667, 0.4784],
          [0.0980, 0.0627, 0.1922,  ..., 0.4627, 0.4706, 0.4275],
          ...,
          [0.8157, 0.7882, 0.7765,  ..., 0.6275, 0.2196, 0.2078],
          [0.7059, 0.6784, 0.7294,  ..., 0.7216, 0.3804, 0.3255],
          [0.6941, 0.6588, 0.7020,  ..., 0.8471, 0.5922, 0.4824]],

         [[0.2431, 0.1804, 0.1882,  ..., 0.5176, 0.4902, 0.4863],
          [0.0784, 0.0000, 0.0314,  ..., 0.3451, 0.3255, 0.3412],
          [0.0941, 0.0275, 0.1059,  ..., 0.3294, 0.3294, 0.2863],
          ...,
          [0.6667, 0.6000, 0.6314,  ..., 0.5216, 0.1216, 0.1333],
          [0.5451, 0.4824, 0.5647,  ..., 0.5804, 0.2431, 0.2078],
          [0.5647, 0.5059, 0.5569,  ..., 0.7216, 0.4627, 0.3608]],

         [[0.2471, 0.1765, 0.1686,  ..., 0.4235, 0.4000, 0.4039],
          [0.0784, 0.0000, 0.0000,  ..., 0.2157, 0.1961, 0.2235],
          [0.0824, 0.0000, 0.0314,  ..., 0

In [ ]:
data.shape # 축 : 4개 / 0번 : 50000 / 1번 : 3 / 2번 : 32 / 3번 : 32 -> 연산기준 : 1번축

torch.Size([50000, 3, 32, 32])

In [ ]:
print(data.mean(dim=[0,2,3])) # 채널당 평균값
print(data.std(dim=[0,2,3])) # 채널 분산(표준편차)

tensor([0.4914, 0.4822, 0.4465])
tensor([0.2470, 0.2435, 0.2616])


In [21]:
# 상수 설정
BATCH_SIZE=64
EPOCHS=10

In [22]:
# 데이터 수정 (노이즈 삽입)
# 1. 데이터 준비
transform=transforms.Compose([
    # transforms.RandomHorizontalFlip(), # 데이터 증강(노이즈삽입)
    transforms.ToTensor(),#입력 데이터 정리
    transforms.Normalize((0.4914, 0.4822, 0.4465),(0.2470, 0.2435, 0.2616))
]) # 데이터 사용 방식 내용 결정
tr_ds_loader=torch.utils.data.DataLoader(
    datasets.CIFAR10('./data/',
        train=True,
        download=False,
        transform=transform
    ),
    batch_size=BATCH_SIZE,
    shuffle=True
)
tt_ds_loader=torch.utils.data.DataLoader(
    datasets.CIFAR10('./data/',
        train=False,
        transform=transform
    ),
    batch_size=BATCH_SIZE,
    shuffle=True
)

In [27]:
# 2. 모델 설계
class BasicBlock(nn.Module): # 블록 단위 -> 모델 완성x
    def __init__(self,in_planes,planes,stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1=nn.BatchNorm2d(planes)
        
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2=nn.BatchNorm2d(planes)

        self.shortcut=nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

            
    def forward(self,x):
        out=F.relu(self.bn1(self.conv1(x)))
        out=self.bn2(self.conv2(out))
        out+=self.shortcut(x)

        out=F.relu(out)
        return out
        
class Resnet_model(nn.Module):
    def __init__(self,class_n):
        super().__init__()
        self.in_planes=16
        self.conv1=nn.Conv2d(3,16,kernel_size=3,stride=1,padding=1,bias=False)
        self.bn1=nn.BatchNorm2d(16)
        self.l1=self.make_l(16,2,1)
        self.l2=self.make_l(32,2,2)
        self.l3=self.make_l(64,2,2)
        self.out_l=nn.Linear(64,class_n)

    def make_l(self,planes,num_blocks,stride):
        strides=[stride]+[1]*(num_blocks-1)
        l=[]
        for i in strides:
            l.append(BasicBlock(self.in_planes,planes,i))
            self.in_planes=planes
        return nn.Sequential(*l)
    
    def forward(self,x):
        x=F.relu(self.bn1(self.conv1(x)))
        x=self.l1(x)
        x=self.l2(x)
        x=self.l3(x)

        x=F.avg_pool2d(x,8)
        x=x.view(x.size(0),-1)
        out=self.out_l(x)
        return out

In [28]:
r=Resnet_model(10).to(DEVICE)
opt=optim.SGD(r.parameters(),lr=0.1,momentum=0.9,weight_decay=0.0005)
scheduler=optim.lr_scheduler.StepLR(optimizer=opt,step_size=3,gamma=0.1)

In [29]:
def train(m,tr_ds_loader,opt,epoch):
    m.train()
    for i,(x,y) in enumerate(tr_ds_loader):
        data,target=x.to(DEVICE),y.to(DEVICE)
        opt.zero_grad()
        py=m(data)
        loss=F.cross_entropy(py,target)
        loss.backward()
        opt.step()

        if i%100==0:
            print(f"train epoch{epoch} {i*len(data)}/{len(tr_ds_loader.dataset)}({i*100/len(tr_ds_loader)}%) loss:{loss.item()}")
        

In [30]:
@torch.no_grad()#서식
def evaluate(m,tt_ds_loader):
    m.eval()
    test_loss=0
    correct=0
    for data,target in tt_ds_loader:
        data,target=data.to(DEVICE),target.to(DEVICE)
        py=m(data)
        test_loss+=F.cross_entropy(py,target,reduction='sum').item()
        pred=py.max(1,keepdim=True)[1]
        correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss/=len(tt_ds_loader.dataset)
    test_accuracy= correct/len(tt_ds_loader.dataset)*100.
    return test_loss,test_accuracy   

In [31]:
for i in range(1,EPOCHS+1):
    train(r,tr_ds_loader,opt,i)
    scheduler.step()
    test_loss,test_acc=evaluate(r,tt_ds_loader)

    print(f'{i}epoch test_loss:{test_loss},test_accuracy:{test_acc}')

train epoch1 0/50000(0.0%) loss:2.369304895401001
train epoch1 6400/50000(12.787723785166241%) loss:1.744459629058838
train epoch1 12800/50000(25.575447570332482%) loss:1.504948616027832
train epoch1 19200/50000(38.36317135549872%) loss:1.4173904657363892
train epoch1 25600/50000(51.150895140664964%) loss:1.5569417476654053
train epoch1 32000/50000(63.9386189258312%) loss:1.2809737920761108
train epoch1 38400/50000(76.72634271099744%) loss:1.3292120695114136
train epoch1 44800/50000(89.51406649616368%) loss:1.192624568939209
1epoch test_loss:1.3783417663574218,test_accuracy:50.38
train epoch2 0/50000(0.0%) loss:1.157811164855957
train epoch2 6400/50000(12.787723785166241%) loss:1.1426254510879517
train epoch2 12800/50000(25.575447570332482%) loss:1.007771611213684
train epoch2 19200/50000(38.36317135549872%) loss:0.9437767267227173
train epoch2 25600/50000(51.150895140664964%) loss:0.8931344747543335
train epoch2 32000/50000(63.9386189258312%) loss:1.0528738498687744
train epoch2 38400